# Optimization Engine — quickstart

This notebook walks through the full pipeline:
1. Load (or generate) prices.
2. Configure the optimizer.
3. Run several techniques and compare.
4. Build the efficient frontier.
5. Backtest and export to Excel.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd

from optimization_engine.data.loader import sample_dataset, prices_to_returns
from optimization_engine.config import EngineConfig, OptimizerSpec
from optimization_engine.engine import run_engine
from optimization_engine.reporting.plots import (
    plot_efficient_frontier, plot_portfolio_composition,
    plot_risk_contributions, plot_correlation_heatmap, plot_wealth_index,
)
from optimization_engine.optimizers.factory import available_optimizers
available_optimizers()

In [ ]:
prices = sample_dataset(n_periods=252 * 8)
returns = prices_to_returns(prices)
returns.tail()

In [ ]:
groups = {
    'US_Equity': 'Equity', 'Intl_Equity': 'Equity', 'EM_Equity': 'Equity',
    'Real_Estate': 'Alternatives', 'Commodities': 'Alternatives',
    'Infra': 'Alternatives', 'Gold': 'Alternatives',
    'US_Treasuries': 'FixedIncome', 'TIPS': 'FixedIncome',
    'IG_Credit': 'FixedIncome', 'HY_Credit': 'FixedIncome',
    'EM_Debt': 'FixedIncome', 'Cash': 'FixedIncome',
}
expected = {
    'US_Equity': 0.08, 'Intl_Equity': 0.07, 'EM_Equity': 0.09,
    'Real_Estate': 0.06, 'Commodities': 0.04, 'Infra': 0.07, 'Gold': 0.04,
    'US_Treasuries': 0.03, 'TIPS': 0.03, 'IG_Credit': 0.04,
    'HY_Credit': 0.05, 'EM_Debt': 0.05, 'Cash': 0.025,
}
bounds = {a: [0.0, 0.5] for a in expected}
group_bounds = {'Equity': [0.30, 0.70], 'Alternatives': [0.0, 0.20], 'FixedIncome': [0.20, 0.60]}

def cfg(method, **kwargs):
    return EngineConfig(
        expected_returns=expected, bounds=bounds, groups=groups,
        group_bounds=group_bounds,
        optimizer=OptimizerSpec(name=method, risk_free_rate=0.04, **kwargs),
    )

In [ ]:
comparisons = {}
for method, kwargs in [
    ('mean_variance', {'target_return': 0.07}),
    ('min_variance', {}),
    ('max_sharpe', {}),
    ('risk_parity', {}),
    ('hrp', {}),
    ('max_diversification', {}),
    ('inverse_vol', {}),
    ('equal_weight', {}),
]:
    run = run_engine(returns, cfg(method, **kwargs))
    comparisons[method] = run.result.weights

comparison_df = pd.concat(comparisons, axis=1)
comparison_df.round(3)

In [ ]:
frontier_run = run_engine(
    returns,
    cfg('mean_variance'),
    build_frontier=True, n_frontier_points=25,
)
plot_efficient_frontier(
    frontier_run.frontier.summary, frontier_run.frontier.max_sharpe_index
)

In [ ]:
plot_portfolio_composition(frontier_run.frontier.weights)

In [ ]:
plot_correlation_heatmap(returns.corr())